<div style="display: flex; align-items: center;">
  <img src="../Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Premortem (What-If) Decision Analysis  </span>
</div>

> 📜 **_"The pre-mortem method has a number of benefits besides identifying trouble spots in a plan – reducing overconfidence and promoting discoveries. The method can strengthen members’ mental models… And it can help create a culture of candor and trust."_**
>
> -- <cite>Gary Klein (Psychologist, creator of the Pre-mortem method)</cite>  

Decision Intelligence applied in this module:  
* Using Premortem Decision Analysis to improve decision quality & outcomes  
* Using a Premortem decision output for improving of Gathering Intelligence 
* Attain different perspectives using mutliple decision frames  
* Apply a Decision Framework to Premortem Decision Analysis 


### What is a Premortem (What-If Analysis)  

You have probably heard of a **postmortem (post-mortem)**, which is a look back at an actual failure (or success) to understand why it happened and learn from it. In the context of decision making, after the decision outcome is clear you can analyze why the result occured. For example, a hedge fund may host a postmortem meeting after analyzing an unsuccesful move on a tech stock that didn't materialize the return on investment as was planned. 

In a **premortem (pre-mortem)** the outcome is "imagined" and then it is figured out why that potential "imagined" outcome happened. In a premortem, a planning exercise is held where a team imagines that a project or decision has already succeeded or failed. The team then works backward, brainstorming what might have caused things to go wrong. This approach allows everyone to spot potential risks and address them before starting the project. Therefore, it’s a bit like predicting the future and using that prediction to make the forecast stronger. 

### Example Decision Scenario - Purchasing a Car   

Imagine a decision scenario, where you are thinking about buying a new car. Let's play a game of "What-If" that the purchase decision results in either two outcomes a: **disaster** or a **successful** outcome. 

#### Premortem Outcome - Disaster  

<img style="display: block; margin: auto;" width ="700px" src="../Images/Scenarios/Scenario-DecisionPremortem.png">

In a **premortem (pre-mortem)**, you pretend the car purchase turned out to be a disaster, which resulted in a highly undesirable decision outcome. Now, you ask yourself:  
**What went wrong with the decision to buy a new car?**

Here are some potential reasons the car purchase had a poor outcome:  

* **Financial issues:** You realize later you can’t comfortably afford the monthly payments or maintenance costs.  
* **Unexpected repairs:** The car has hidden mechanical problems, leading to costly repairs.  
* **Insurance costs:** Insurance is much higher than expected, impacting your budget.
* **Poor fuel efficiency:** The car uses more gas than you thought, making it expensive to drive.
Resale value drop: The car’s value plummets quickly, leaving you with less if you want to sell or trade it.  

#### Premortem Outcome - Success

In a premortem for a successful car purchase, imagine you bought the car, and it turned out to be a succssful decision. Now, you ask yourself:  
**What went right with the decision to buy a new car?**  

Here are some potential reasons the car purchase had a good outcome: 

* **Affordable and realistic budget:** You set a budget that allowed for comfortable monthly payments and covered insurance, fuel, and maintenance without straining your finances.  
* **Thorough inspection:** You got a detailed inspection before buying, which confirmed the car was in excellent condition and minimized the risk of future repairs.
* **Reliable model choice:** You researched models with a reputation for reliability, so you experience fewer repairs and smooth performance.
* **Fuel efficiency matches needs:** You chose a car that meets your fuel efficiency needs, saving money on gas and fitting well with your daily driving habits.   
* **High resale value:** You picked a make and model that holds its value well, giving you confidence that it’s a good long-term investment.  

#### Premortem (What-If) Decision Analysis 

Imagining successful outcomes helps you focus on actions like sticking to a car budget, getting a full inspection, and choosing a reliable model. You will be more likely to make a purchase you’ll be happy with. By plyaing a "game" of What-If with decision criteria in advance, you could adapt your strategy. For example, you could consider to thoroughly check maintenance records, buy additional warranty, test drive with similar terrain in your area, or reassess your budget to avoid these potential problems.

How can Generative AI help? Using Generative AI, the LLM models can be used to "imagine" premortem outcomes and  brainstorm potential factors why that certain outcome occured. Generative AI agents can act as your "premortem" team. This can lead to additional questions, gathering of intelligence, checklists, negotiation leverage etc. you didn't consider initially. 

---
### Step 1 - Initialize Configuration Builder & Build the AI Orchestration

In [1]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.12"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.12"
#r "nuget: System.Text.Json, 10.0.12"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.12 Microsoft.Extensions.Configuration.Json, 10.0.12 System.Text.Json, 10.0.12

In [2]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.10.0"

// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.13.0"

using Azure;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient instance
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey!);

    var azureOpenAIClient = new OpenAIClient(
        apiKeyCredential,
        new OpenAIClientOptions
        {
            Endpoint = new Uri($"{azureOpenAIEndpoint!.TrimEnd('/')}/openai/v1")
        });

    #pragma warning disable OPENAI001
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    #pragma warning disable OPENAI001
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

Installed Packages Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.10.0 Microsoft.Extensions.AI.Abstractions, 10.10.0 Microsoft.Extensions.AI.OpenAI, 10.10.0 OpenAI, 2.13.0

Using Azure OpenAI Service


----
### Step 2 - Premortem Risk Analysis with AI Reasoning Models

> 📜 **_"Success is a lousy teacher. It seduces smart people into thinking they can’t lose."_**
>
> -- <cite>Bill Gates (Co-Founder of Microsoft, Philantropist)</cite>  

In the decision scenario below, an AI model will perform a premortem (What-If) analysis on the decision whether to purchase or lease a car. The AI model will provide risks and items to consider for each of the decision options: purchasing a car or leasing a car. 

In [3]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForDecisions = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and nothing else. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";
var decisionPreMortemAnalysisPrompt = """
Perform an advanced premortem (What-If) analysis on the following decision scenario:
You are deciding to purchase a new car versus leasing a car.
""";

// Create a new pre-mortem analysis chat
var chatMessagesDecisionPremortem = new List<ChatMessage>();
// Add system and user messages to the chat history
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForDecisions);
var userMessage = new ChatMessage(ChatRole.User, decisionPreMortemAnalysisPrompt);
chatMessagesDecisionPremortem.Add(systemMessage);
chatMessagesDecisionPremortem.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var chatMessagesDecisionPremortemResponseText = chatMessagesDecisionPremortemResponse.Text;

// Display the response string as Markdown
chatMessagesDecisionPremortemResponseText.DisplayAs("text/markdown");

// Add the response from the model to the chat history
var assistantMessage = new ChatMessage(ChatRole.Assistant, chatMessagesDecisionPremortemResponseText);
chatMessagesDecisionPremortem.Add(assistantMessage);

| Premortem area | Purchase a new car: plausible failure | Lease a car: plausible failure | Early warning indicators | Preventive controls and decision rules |
|---|---|---|---|---|
| Core assumption | You keep the car long enough for ownership to overcome higher upfront cost, but sell or trade after only a few years. | You value lower initial commitment, but continue leasing repeatedly and never build an ownership asset. | Expected ownership period changes; frequent vehicle upgrades; dissatisfaction appears within 24 months. | Estimate total cost over the actual expected holding period, not an assumed long-term period. Compare three-year, six-year, and ten-year outcomes. |
| Total cost | Depreciation is underestimated, financing interest is ignored, or maintenance and taxes are excluded from the comparison. | The advertised monthly payment obscures acquisition fees, taxes, disposition fees, excess mileage, wear charges, and repeated down payments. | Dealer quote omits a full out-the-door or all-in cost; comparison focuses mainly on monthly payment. | Use total cash outflow and present-value cost. Include down payment, taxes, fees, insurance, fuel, maintenance, financing, residual value, and end-of-term charges. |
| Depreciation and resale | The model depreciates faster than expected because of weak demand, recalls, changing technology, or an unpopular body style. | The contracted residual value is unfavorable, making the lease expensive relative to the vehicle’s actual market value. | Comparable used-car prices fall; incentives increase on the same model; industry demand weakens. | Compare multiple models and residual assumptions. For a purchase, stress-test resale value at 20% to 30% below estimate. For a lease, compare the residual percentage with independent market forecasts. |
| Financing risk | Interest rates remain high, the loan term is extended to make payments affordable, and the vehicle becomes financially upside down. | A large capitalized-cost reduction or down payment is lost if the vehicle is stolen or totaled early, despite insurance reimbursement. | Loan term exceeds five or six years; negative equity is rolled into the new transaction; lease requires substantial upfront cash. | Prefer a modest purchase down payment and shorter loan term. Avoid large lease down payments; use gap protection and confirm how insurance settlement works. |
| Cash-flow resilience | A job loss, income reduction, or major expense makes the payment and ownership costs unaffordable. | The fixed lease obligation becomes difficult to exit because early termination is costly and transfer options are limited. | Emergency savings falls below three to six months; income is volatile; debt-service ratio rises. | Test affordability under a 20% income reduction. Preserve emergency liquidity. Do not use a lease or loan payment that depends on bonuses or overtime. |
| Mileage uncertainty | High mileage reduces resale value and increases wear, but the owner bears the risk rather than paying a contractual penalty. | Actual driving exceeds the mileage allowance, creating substantial per-mile charges at lease end. | Commute, travel, family, or work patterns are changing; annual mileage is not tracked. | Track actual mileage for at least six months. Purchase is safer when mileage is highly uncertain or likely to be high. If leasing, buy sufficient mileage upfront and understand overage rates. |
| Vehicle condition | Accidents, cosmetic damage, neglected maintenance, or harsh use reduce resale value and create lower-than-expected trade-in proceeds. | Normal use is judged as excess wear, producing end-of-lease charges for tires, dents, glass, paint, or interior damage. | Frequent parking damage; pets, children, construction work, off-road use, or poor maintenance habits. | Obtain the lessor’s wear-and-use standards. Budget for tires and repairs. Purchase is generally more forgiving for rough use, but resale assumptions should still reflect condition. |
| Flexibility and exit | Selling early produces a loss because transaction costs and depreciation exceed remaining loan equity. | Early termination, lease transfer, or buyout is expensive or restricted by contract and credit requirements. | Life change is plausible within two to three years; relocation, family growth, or uncertain employment. | If flexibility is valuable, compare a lightly used purchase, shorter loan, or lease-transfer-friendly contract. Read termination, transfer, and buyout clauses before signing. |
| Reliability and repairs | The vehicle experiences major repairs after warranty expiration, creating unpredictable ownership costs. | Repairs may be covered under warranty, but the lessee remains responsible for maintenance, downtime, and non-covered damage. | Model has poor reliability history; warranty is short; service network is weak. | Check independent reliability data, warranty scope, maintenance schedule, and repair pricing. A lease is more attractive when reliability uncertainty is high and warranty coverage is comprehensive. |
| Technology and obsolescence | Rapid advances in batteries, driver assistance, infotainment, or efficiency make the purchased vehicle feel outdated and reduce resale value. | Frequent leasing limits long-term obsolescence risk, but lease pricing can incorporate expected technology risk and may prevent equity accumulation. | New technology releases are accelerating; incentives or regulations may change; charging standards are evolving. | Lease when technology uncertainty is strategically important and the premium is acceptable. Purchase when the vehicle’s technology is mature and long-term utility is clear. |
| Electric-vehicle-specific risk | Battery degradation, charging limitations, incentive changes, or rapid model improvements reduce usefulness and resale value. | Residual assumptions may be unfavorable, and battery-related lease terms, warranty exclusions, or end-of-term rules may be unclear. | Charging access is uncertain; annual mileage is high; incentives are changing; battery warranty terms are misunderstood. | Verify home and public charging economics, battery warranty, degradation coverage, tax-credit treatment, residual value, and lease-end purchase terms. |
| Insurance and operating cost | Insurance, registration, tires, fuel, and maintenance are higher than forecast, making ownership materially more expensive. | Leasing often requires higher coverage limits and lower deductibles, increasing recurring insurance cost. | Insurance quote differs substantially by vehicle; tire replacement interval is short; fuel economy assumptions are optimistic. | Obtain insurance quotes before deciding. Use realistic local fuel, tire, registration, and maintenance costs rather than manufacturer estimates. |
| Contract and legal risk | Dealer add-ons, documentation fees, service contracts, or financing conditions increase the amount financed without improving value. | Contract language creates unexpected charges for disposition, modifications, early termination, wear, mileage, or insurance. | Sales representative emphasizes payment rather than price; terms are explained verbally but not documented. | Secure written itemized offers from multiple dealers. Review every fee, cap-cost reduction, money factor, residual, mileage allowance, and end-of-term obligation. |
| Dealer and execution risk | Trade-in value is overstated initially, then reduced after inspection; promised incentives or financing terms disappear at signing. | Lease incentive, money factor, residual, or loyalty discount is changed during negotiation. | Last-minute revisions; pressure to sign immediately; inconsistent figures across documents. | Separate vehicle price, trade-in, financing, and incentives. Verify the final contract independently. Be prepared to walk away. |
| Opportunity cost | Cash used for a down payment or purchase could have funded investments, debt repayment, or an emergency reserve. | Lower upfront cash appears efficient, but repeated lease payments may produce worse long-term wealth accumulation. | Alternative investment return or high-interest debt rate exceeds the vehicle financing cost. | Compare after-tax investment returns and debt costs. Avoid using cash for the car when high-interest debt or inadequate emergency savings exists. |
| Behavioral failure | Ownership encourages keeping an unsuitable car too long because selling feels like admitting a loss, or encourages expensive customization. | Leasing encourages frequent upgrades, payment anchoring, and acceptance of perpetually higher transportation spending. | New-car cycle shortens; accessories and upgrades exceed plan; decision is driven by status or novelty. | Establish a written holding-period and transportation-budget policy. Evaluate the decision by total cost and utility, not payment or emotional appeal. |
| Equity and end-of-term outcome | The vehicle’s equity is lower than expected, so trading it in does not meaningfully reduce the next purchase cost. | There is no equity unless the vehicle’s market value exceeds the contractual buyout, and the lessee may overlook that option. | Market value approaches or falls below payoff or buyout amount; dealer offers are materially below expectations. | Obtain independent payoff, trade-in, and purchase-option valuations before acting. For a lease, compare buyout price plus taxes and fees with market value and expected ownership cost. |
| Scenario stress test | High depreciation, high interest, high maintenance, and early sale combine to make ownership substantially worse than planned. | High mileage, excess wear, early termination, and repeated leasing combine to make leasing substantially worse than planned. | More than one adverse assumption begins occurring simultaneously. | Model adverse, base, and favorable cases. Choose the option with acceptable downside, not merely the lowest base-case cost. |
| Worst-case financial exposure | The car is totaled or sold while loan balance exceeds market value, requiring additional cash to exit. | The car is totaled, stolen, or surrendered early and insurance does not fully cover contractual obligations or fees. | Loan-to-value exceeds 100%; gap coverage is absent or poorly understood; upfront lease payment is large. | Maintain gap coverage where relevant, minimize negative equity, avoid large lease prepayments, and verify insurer and lessor settlement procedures. |
| Decision-quality failure | The decision is made from monthly payment, dealer claims, or an optimistic resale estimate rather than comparable all-in economics. | The decision is made from a low advertised payment without recognizing mileage, fees, restrictions, and lease-cycle costs. | No written assumptions; only one quote; no independent resale or insurance check. | Use a standardized worksheet with identical assumptions, at least three quotes, independent market data, and a documented sensitivity analysis. |
| Conditions favoring purchase | Long ownership horizon, high or uncertain mileage, rough use, desire for customization, strong cash or financing position, and willingness to bear resale and repair risk. | These conditions make leasing less attractive because restrictions and repeated contracts have high value costs. | Expected holding period exceeds roughly five to seven years; mileage is high; vehicle is likely to be kept after payoff. | Prefer purchasing when durable ownership utility and long-term equity matter more than short-term flexibility. |
| Conditions favoring leasing | These conditions make leasing less attractive because depreciation and repair risk remain with the owner. | Short predictable holding period, stable low mileage, strong warranty coverage, attractive residual, desire for newer vehicles, and high technology uncertainty. | Expected use is two to four years; annual mileage is predictable; contract has low upfront cash and transparent terms. | Prefer leasing when flexibility, warranty coverage, and obsolescence protection justify the premium over ownership. |
| Reconsideration triggers | Reconsider purchase if the loan requires an extended term, negative equity is rolled in, ownership horizon shortens, or reliability and resale outlook deteriorate. | Reconsider lease if mileage becomes uncertain, upfront cash is large, early exit is plausible, residual is weak, or repeated leasing exceeds purchase cost. | Any major change in income, commute, family needs, tax treatment, incentives, or vehicle availability. | Recalculate the decision when assumptions change rather than defending the original choice. |
| Final premortem conclusion | Purchase fails mainly through underestimated depreciation, financing cost, repairs, resale loss, and premature exit. | Lease fails mainly through underestimated all-in fees, mileage and wear charges, exit restrictions, weak residual economics, and perpetual payments. | The strongest warning sign is reliance on a low monthly payment without a full-cost and downside analysis. | Select the option whose worst credible scenario remains financially tolerable and operationally useful, even if its base-case cost is not the lowest. |

The large reasoning model provided a comprehensive set of criteria, risks, mitigation strategy and approaches for the two decisions. This is a great option, but it might be more ideal to provide this in an easy to consume table (checklist) for the user that is easy to compare and contrast. This What-If analysis is easy to consume on a mobile device or even print out and annotate. 

This Markdown table can be further simplified by highlighting which risk factor is more favorable when comparing a car purchase versus a car lease. This can make the decision process much easier to the user, as at a glance they can see overwhelming evidence that favors once decision option over the other.  

In [4]:
var simplifyRiskAnalysisPrompt = """
You are provided a detailed markdown table in the above chat history that 
compares Purchase and Lease options across various risk factors and uncertainties. 
Your task is to convert this table into a simplified markdown table that uses check marks (☑️) 
to indicate which option is less risky or more ideal for each risk factor. 
The output table should have three columns: "Risk Factor / Uncertainty / Potential Outcome", "Purchase", and "Lease". 
For each risk factor, decide based on the detailed descriptions which option (Purchase or Lease) 
is preferable (i.e., less risky or more advantageous) and place a check mark (☑️) in that column. 
If an option is not the ideal choice for a given risk factor, leave its cell blank. 
Make sure the simplified table closely reflects the key insights from the detailed table. 

Provide a small summary below the table of the key insights 
from the detailed table in a single paragraph before the simplified table.
""";

var simplifyRiskAnalysisMessage = new ChatMessage(ChatRole.Assistant, simplifyRiskAnalysisPrompt);
chatMessagesDecisionPremortem.Add(simplifyRiskAnalysisMessage);

// Execute the chat messages against the AI model
var simplifiedDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var simplifiedDecisionPremortemResponseText = simplifiedDecisionPremortemResponse.Text;

// Display the response string as Markdown
simplifiedDecisionPremortemResponseText.DisplayAs("text/markdown");


| Risk Factor / Uncertainty / Potential Outcome | Purchase | Lease |
|---|---|---|
| Long ownership horizon | ☑️ |  |
| Short, predictable ownership period |  | ☑️ |
| High or uncertain mileage | ☑️ |  |
| Low, predictable mileage |  | ☑️ |
| Rough use, pets, children, or frequent cosmetic damage | ☑️ |  |
| Desire to customize the vehicle | ☑️ |  |
| Avoiding long-term repair risk |  | ☑️ |
| Strong reliability and long warranty coverage |  | ☑️ |
| Building long-term equity | ☑️ |  |
| Lower upfront cash requirement |  | ☑️ |
| Avoiding perpetual vehicle payments | ☑️ |  |
| Frequent vehicle upgrades |  | ☑️ |
| Protection from technology obsolescence |  | ☑️ |
| Uncertain future employment or relocation | ☑️ |  |
| Need for maximum exit flexibility | ☑️ |  |
| Predictable monthly transportation cost |  | ☑️ |
| Risk of high depreciation |  | ☑️ |
| Risk of excess mileage and wear charges | ☑️ |  |
| Risk of being financially upside down |  | ☑️ |
| High interest rates or extended financing |  | ☑️ |
| High-interest debt or inadequate emergency savings |  |  |
| Strong cash position and affordable financing | ☑️ |  |
| Electric-vehicle technology uncertainty |  | ☑️ |
| Uncertain resale value |  | ☑️ |
| Desire to minimize total long-term cost | ☑️ |  |
| Preference for convenience and warranty coverage |  | ☑️ |
| Risk from large upfront payment | ☑️ |  |
| Risk from early termination | ☑️ |  |
| Overall downside tolerance | ☑️ |  |
| Key insights summary | Purchase is generally safer for long ownership, high or uncertain mileage, rough use, flexibility, customization, and long-term cost control. | Lease is generally safer for short predictable use, low mileage, frequent upgrades, warranty protection, technology uncertainty, and reduced exposure to depreciation and repairs. |

### Step 3 - Premortem Analysis for Radio Telescope Operations 

Exploring space with telescopes is a time consuming investment. Current telescope designs are limited to exploring small areas of space in a certain time frame. Without going into details, the process is much more complex than snapping a picture with a mobile device. Telescopes can be pointed at a specific space area for days, weeks or even more than months! If a telescope is exploring a certain area of space it isn't collecting data on other areas. Scientists and researchers compete for valuable access to telescope resources by providing observatories what space area they want to explore, any cellestial contraints, timeframes, calibration costs, data collection transfer etc. This presents a very interesting Decision Intelligence constraints problem that observatories must manage. 

<img style="display: block; margin: auto;" width ="700px" src="../Images/Scenarios/Scenario-RadioTelescope.png">

#### Decision Scenario - Premortem for Radio Telescope Operations 

> 📜 "We choose to go to the Moon in this decade and do the other things, not because they are easy, but because they are hard." 
>
> -- <cite>John F. Kennedy (35th President of the United States)</cite>  

**Intro to Decision Scenario:** A newly commissioned radio telescope stands on the verge of unveiling the proof of alien life, but it is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. Faced with the risk of hardware damage or data corruption from running the telescope prematurely, scientists debate whether to switch it on now or wait until everything is fully operational, knowing they could lose a once-in-a-generation opportunity. Determined not to squander this brief cosmic window, the team conducts a premortem analysis to predict what might go wrong. Armed with these insights, they can better prepare for potential pitfalls and strengthen their chances of catching historic signals from across the galaxy.

**Pre-Mortem Risk Scenario:** Let's assume (pretend) the scientists decide to rush the radio telescope to operate early to find alien life. Furthermore, assume a that this decision has led to a disaster with hardware failure to the telescope. Even though the (What-If) decision was made to rush the radio telescope operationally, formal scientific telescope operations have been followed. The goal of this premortem (What-If analysis) is to illuminate any risk mitigation strategies and craft a risk mitigation approach that may have been uncovered based on the premortem. 

Techinques like this a very powerful in a variety of business scenarios, where existing operational frameworks exist but simulating an outcome can uncover gaps, emerging risks and other opportunities to optimize. 

In [5]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForPremortemRadioTelescope = """
You are a Decision Intelligence Assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving.
Apply systems thinking to various scenarios. Provide structured, logical, and comprehensive advice.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and a summary below if asked. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";

var decisionAnalysisPromptForPremortemRadioTelescope = """
You are a project manager at a new radio telescope facility looking for alien life.  
It is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, 
an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. 
Faced with the risk of hardware damage or data corruption from running the telescope prematurely, 
scientists debate whether to switch it on now or wait until everything is fully operational—knowing 
they could lose a once-in-a-generation opportunity. 

Imagine it is six months from now and the project to detect potential alien signals from the sudden cosmic event has failed. 
The radio telescope was likely activated prematurely, software remained untested, and the team rushed to capture transmissions. 
Despite the urgency, the outcome was disastrous: hardware suffered damage under the increased load, 
data came out corrupted, or no meaningful signals were confirmed. 

Now, step into that moment of failure—what specific issues, oversights, or cascading errors caused this grim scenario?

1) Envision every possible way the radio telescope's early deployment could have led to compromised or unusable data.
2) Identify how incomplete software testing and rushed calibration might have 
contributed to errors in processing or storing signals. 
3) Consider any overlooked safety protocols or hardware precautions that might have led to damage or malfunctions.
4) Examine the roles and responsibilities of the team and think about communication breakdowns, 
rushed decisions, and missed signals or deadlines.
5) Assess how time pressure from the rare cosmic event might have amplified smaller mistakes and 
turned them into systemic failures.

Use the outcomes of this “what if” vision to create a detailed list of preventive measures. 
Then, propose fail-safes, additional resources, or procedural changes to ensure that, 
should you choose to activate the telescope early, you can still capture the event safely and 
effectively—maximizing the chance to detect historic signals without jeopardizing the entire operation.

Create a summary of the key insights from the detailed analysis below.
""";

// Create a new pre-mortem analysis chat
var chatMessagesPreMortemTelescopeAnalysis = new List<ChatMessage>();
// Add system and user messages to the chat messages for the pre-mortem analysis of the radio telescope scenario
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForPremortemRadioTelescope);
var userMessage = new ChatMessage(ChatRole.User, decisionAnalysisPromptForPremortemRadioTelescope);
chatMessagesPreMortemTelescopeAnalysis.Add(systemMessage);
chatMessagesPreMortemTelescopeAnalysis.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesPreMortemTelescopeAnalysisResponse = 
    await chatClient.GetResponseAsync(chatMessagesPreMortemTelescopeAnalysis);
var chatMessagesPreMortemTelescopeAnalysisResponseText = chatMessagesPreMortemTelescopeAnalysisResponse.Text;

// Display the response string as Markdown
chatMessagesPreMortemTelescopeAnalysisResponseText.DisplayAs("text/markdown");

| Area | Plausible failure scenario six months later | Preventive measures | Early-activation fail-safes and resources | Decision indicators and accountability |
|---|---|---|---|---|
| Mission governance | The team activated the telescope without a documented risk decision, success criteria, or authority boundaries. Different groups assumed someone else had accepted the risk. | Establish a written early-activation charter defining objectives, permitted operating limits, decision rights, stop conditions, and evidence required for escalation. | Appoint an incident commander with authority to pause or shut down operations. Use a two-person approval rule for every increase in power, bandwidth, observing duration, or configuration complexity. | The project director owns the final go or no-go decision; the incident commander owns real-time safety decisions; subsystem leads own technical readiness evidence. |
| Premature deployment scope | The team attempted full scientific operations instead of a constrained campaign, overloading hardware, software, and personnel simultaneously. | Separate the mission into minimum viable observation, engineering validation, and full-science phases. Define the smallest useful dataset that could preserve the opportunity. | Begin with passive monitoring, reduced gain, narrow bandwidth, short observation windows, and redundant low-risk recording paths. Expand only after predefined stability checks pass. | Any request to exceed the approved operating envelope requires a documented risk review and explicit approval. |
| Incomplete calibration | Gain, timing, pointing, frequency response, polarization, noise temperature, and interference characteristics were insufficiently calibrated, making real signals indistinguishable from instrumental artifacts. | Maintain calibration baselines, reference-source observations, environmental measurements, and uncertainty budgets for every instrument mode. | Interleave short calibration scans with event observations. Preserve raw calibrator data and record calibration state with every science file. | The calibration lead reports confidence ranges and known blind spots; data cannot be labeled scientifically usable without calibration provenance. |
| Rushed calibration changes | Engineers changed calibration constants during the event without version control, causing inconsistent data products across observation periods. | Require configuration management, immutable releases, change logs, rollback points, and approval for emergency parameter changes. | Freeze a known-good configuration before the event. Test emergency changes on a parallel or replayed data stream before applying them to live acquisition. | A change is rejected if its effect cannot be measured, reversed, and linked to affected datasets. |
| Software defects | Untested acquisition, signal-processing, scheduling, or control software dropped packets, misordered samples, applied incorrect units, or silently produced plausible but false results. | Use unit, integration, hardware-in-the-loop, load, fault-injection, soak, and replay testing against synthetic and historical datasets. | Run a hardened minimal software path in parallel with the experimental suite. Disable nonessential processing and require explicit health checks before each mode transition. | The software lead signs off only on tested functions; untested features remain disabled or clearly marked as engineering-only. |
| Data corruption | High data rates exceeded storage or network capacity, resulting in packet loss, buffer overruns, truncation, corrupted indexes, or incomplete files. | Calculate end-to-end throughput with safety margins. Test sustained capture, storage failover, metadata integrity, and recovery from interrupted writes. | Use local buffering, independent recorders, chunked immutable files, checksums, queue-depth alarms, and automatic rate reduction before buffer exhaustion. | Automatic shutdown or bandwidth reduction occurs when packet loss, queue depth, latency, or storage headroom crosses a defined threshold. |
| Loss of irreplaceable raw data | The system retained only processed products, discarded unusual frequency ranges, or overwrote raw data during a storage emergency. | Define raw data as the highest-priority retention class. Reserve capacity, enforce write-once or append-only storage, and document retention priorities. | Allocate a protected emergency archive for the highest-value time and frequency windows. Preserve a low-resolution full-spectrum stream if full-resolution capture is impossible. | The data manager verifies that raw, metadata, calibration, and health streams are retained before processing products are generated. |
| Software and hardware incompatibility | Drivers, firmware, timing systems, file formats, or interfaces behaved differently in production than in the test environment. | Establish a hardware and software bill of materials, compatibility matrix, reproducible deployment package, and production-like test environment. | Lock firmware and driver versions. Keep a tested fallback image and maintain an isolated staging system for any update. | No live update is permitted during the event unless it addresses an immediate safety threat and has an approved rollback plan. |
| Timing and synchronization errors | Clock drift, incorrect timestamps, lost leap-second handling, or synchronization failure prevented correlation across receivers and made signal localization impossible. | Validate timing against redundant references and independent clocks. Monitor drift continuously and test behavior during loss of synchronization. | Record raw clock status, timing offsets, synchronization alarms, and observatory time in every data block. Continue recording locally if external timing is temporarily lost. | The timing lead defines maximum acceptable uncertainty; data collected outside that limit is flagged rather than silently accepted. |
| Radio-frequency interference | Local equipment, communications systems, power electronics, satellites, or emergency maintenance created signals that were mistaken for alien transmissions or masked the event. | Complete an interference survey, establish protected observing modes, coordinate site activity, and maintain an interference source catalog. | Restrict nonessential transmitters, use control observations and off-source comparisons, and maintain an independent reference antenna where practical. | Candidate signals require confirmation across independent receivers, observing modes, or geographically separated facilities. |
| Hardware overload | Amplifiers, receivers, digitizers, cooling systems, power supplies, or drives operated beyond thermal, electrical, mechanical, or duty-cycle limits and failed under increased load. | Verify derating, thermal margins, power quality, cooling capacity, component protection, and maximum safe operating envelopes. Test progressively under controlled loads. | Start at reduced gain and duty cycle. Monitor temperature, current, voltage, vibration, fan speed, and error counters with automatic interlocks and staged shutdown. | Hardware protection limits are hard-coded where possible; operators cannot override them without engineering approval and logged justification. |
| Antenna and pointing risk | Uncalibrated pointing, unexpected wind loading, control-loop instability, or incorrect slew commands damaged drives or caused observations to miss the source. | Validate pointing models, slew limits, wind stow logic, drive brakes, encoders, geofencing, and manual recovery procedures. | Use conservative slew rates, fixed-pointing observations first, independent position verification, and automatic stow on wind, vibration, encoder disagreement, or control loss. | The antenna safety lead can stop motion immediately; no experimental control software may bypass certified motion limits. |
| Environmental and facility hazards | Lightning, power fluctuations, overheating, water ingress, dust, vibration, or cooling failure caused outages or component damage during the emergency campaign. | Confirm facility monitoring, surge protection, grounding, backup power, HVAC capacity, fire suppression, and preventive maintenance status. | Use uninterruptible power for control and metadata systems, generator support where available, remote shutdown capability, and weather-triggered operating restrictions. | Facilities staff provide real-time environmental status and predefine conditions requiring reduced operation or shutdown. |
| Single points of failure | One recorder, network switch, clock, power source, operator, or specialist became unavailable and halted the mission or invalidated the dataset. | Conduct a dependency and failure-mode analysis. Remove critical single points through redundancy, spares, cross-training, and documented recovery procedures. | Use independent recording paths, redundant timing, spare storage, hot or cold standby equipment, and an on-call roster covering every critical function. | The operations manager tracks readiness of each redundancy and confirms recovery time objectives before activation. |
| Data provenance failure | Teams could not determine which hardware, software, calibration, or environmental conditions produced each file, preventing validation or reuse. | Make metadata capture automatic and mandatory, including configuration hashes, timestamps, calibration version, operator, health metrics, and quality flags. | Store metadata independently from the main processing pipeline and generate periodic signed operational snapshots. | A dataset is not considered preserved unless its raw data, metadata, configuration, and quality history are co-located and verified. |
| False positive interpretation | Unusual interference, software artifacts, transient hardware behavior, or statistical fluctuations were announced as alien signals, damaging credibility and diverting resources. | Define a candidate-signal protocol with independent review, statistical thresholds, known-artifact checks, reproducibility requirements, and external confirmation criteria. | Maintain a blind analysis stream where feasible, compare against control observations, and preserve all rejected candidates with reasons for rejection. | Scientific leads distinguish detection, candidate, and confirmed signal status; no public claim is made from a single pipeline or instrument. |
| False negative interpretation | Aggressive filtering, incorrect thresholds, saturation, narrow search parameters, or storage limits removed the genuine signal before scientists reviewed it. | Test detection pipelines against injected synthetic signals across varied strengths, drift rates, bandwidths, and time structures. | Preserve broad raw or minimally processed data, run multiple detection methods, and prioritize review of data captured immediately before and after anomalies. | The analysis lead reports sensitivity limits and blind spots rather than presenting absence of evidence as evidence of absence. |
| Human factors and fatigue | Long shifts, unclear handovers, stress, confirmation bias, and excessive workload caused operators to miss alarms, misconfigure systems, or continue after warning signs. | Establish shift limits, paired operations, checklists, mandatory breaks, structured handovers, and a speak-up culture with no penalty for stopping work. | Use a two-person control room, prewritten runbooks, alarm prioritization, decision logs, and relief staffing for the full event window. | Any team member may invoke a pause. The incident commander must acknowledge and disposition every safety-critical concern. |
| Communication breakdown | Astronomy, engineering, software, facilities, and leadership teams used different assumptions about readiness, priorities, or observed anomalies. | Hold cross-functional readiness reviews, maintain a single operational dashboard, define escalation paths, and use a common incident vocabulary. | Conduct frequent briefings, log decisions in one shared system, and require explicit read-back for critical commands and handovers. | Every unresolved issue has an owner, severity, deadline, and escalation path; verbal decisions are recorded immediately. |
| Schedule and deadline failure | Calibration, testing, staffing, or data-transfer milestones slipped, but the event plan was not re-baselined; the team discovered critical gaps during live observation. | Use readiness gates with objective evidence, critical-path tracking, contingency buffers, and a formal process for accepting residual risk. | Set an activation cutoff. If prerequisites are not met, switch to a reduced-scope capture mode rather than improvising full operations. | The project manager reports forecast slippage early; leadership chooses scope reduction, added resources, or delay before the deadline becomes an emergency. |
| Time-pressure escalation | An initial small anomaly led to hurried overrides, broader configuration changes, skipped checks, and normalization of increasingly unsafe conditions. | Predefine escalation levels, stop-work triggers, override authority, and maximum emergency duration. Train through simulations and tabletop exercises. | Use a “stabilize before optimize” rule: freeze changes, protect hardware, preserve raw data, and investigate before increasing performance. | Any safety alarm, unexplained data-quality collapse, or loss of redundancy triggers a controlled pause rather than a performance push. |
| External coordination failure | The team failed to alert other observatories, archive partners, domain experts, or emergency support in time to obtain confirmation or backup capacity. | Prepare contact lists, data-sharing agreements, alert templates, time standards, and secure transfer procedures before the event. | Send preliminary time-critical alerts with uncertainty clearly labeled, share compact candidate products first, and request independent observations. | The scientific coordination lead owns external notification and avoids overstating evidence while maximizing confirmation opportunities. |
| Resource shortfall | Limited compute, storage, bandwidth, specialists, spares, and budget forced unsafe trade-offs and caused the team to prioritize processing over preservation. | Quantify resource demand under best, expected, and worst cases. Prearrange cloud or partner capacity, temporary staffing, spare parts, and transport. | Use priority tiers: safety and raw capture first, health telemetry second, candidate detection third, enrichment processing last. | Resource thresholds trigger scope reduction and external support requests before critical systems become saturated. |
| Recovery and continuity failure | After a crash or shutdown, the team could not restart safely, determine the last valid data, or resume without overwriting evidence. | Test backup restoration, restart sequencing, forensic preservation, incident reporting, and disaster-recovery procedures. | Write-protect failed media, snapshot logs and system state, preserve volatile evidence where feasible, and restart from the last verified checkpoint. | The recovery lead controls restart authorization; no repair or reconfiguration may destroy evidence needed for diagnosis. |
| Ethical and scientific integrity risk | Pressure to produce a historic result caused selective reporting, concealed limitations, or premature public communication. | Require independent review, transparent uncertainty statements, complete audit trails, and preapproved communications procedures. | Separate operational urgency from publication urgency. Release only validated facts, including failures, gaps, and alternative explanations. | The scientific integrity officer or equivalent reviewer must approve any external claim of potential extraterrestrial detection. |
| Activation strategy | The team treated the decision as a binary choice between full activation and complete delay, losing the benefits of staged experimentation. | Use a portfolio of operating modes: safe monitoring, engineering capture, limited science, and full science. Evaluate each by expected information value and downside risk. | Activate only the lowest-risk mode that can preserve the event, with explicit upgrade gates based on live evidence. | Choose the mode that maximizes expected scientific value subject to hard safety and data-integrity constraints. |
| Immediate pre-activation gate | The telescope was switched on despite unresolved safety, data, staffing, or recovery deficiencies. | Require a signed readiness checklist covering hardware, software, calibration, storage, timing, interference, staffing, communications, and rollback. | Do not activate unless minimum conditions are met: protected hardware limits, verified raw-data path, health telemetry, shutdown authority, and tested recovery route. | A failed hard gate means no activation; soft gaps require documented mitigation, owner, deadline, and acceptance authority. |
| Core operating model | The campaign optimized for maximum signal sensitivity but neglected reversibility, observability, and preservation of evidence. | Design operations around bounded experiments, independent monitoring, reversible changes, and continuous quality assessment. | Capture raw data and health telemetry continuously, process copies rather than originals, and review quality at fixed intervals. | The primary success measure is preserved, interpretable evidence without unacceptable equipment risk; detection is a secondary outcome. |
| Key insights summary | The likely root cause was not one technical defect but a coupled system failure: urgency weakened governance, incomplete testing increased uncertainty, resource limits amplified data loss, and fatigue degraded judgment. | Treat readiness as a system property rather than a collection of subsystem approvals. Use staged deployment, independent validation, redundancy, provenance, and explicit stop criteria. | Preserve a minimum viable dataset, operate below certified limits, maintain parallel recording and monitoring, freeze changes, and keep a rapid shutdown capability. | The safest high-value choice is usually controlled early activation, not unrestricted activation or passive delay: capture what can be captured, protect the facility, and retain enough evidence for later validation. |